# 03 — Model Training

We visibly instantiate and fit Logistic Regression, KNN, SVM and Random Forest.
Short A/B loops compare identical fitting flights using different feature allowlists.
Scikit-learn handles preprocessing inside each pipeline. **This notebook uses tables
and has no figures.** All settings and thresholds remain the frozen 2025 choices.

In [1]:
from pathlib import Path
import sys
here = Path.cwd()
candidates = [here, *here.parents, here/'departure-delay-classification']
root = next(p for p in candidates if (p/'config/project.json').is_file())
sys.path.insert(0, str(root/'scripts'))
import simple_workflow as project

In [2]:
from time import perf_counter
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

## Define training rows, y and chronological roles

The final refit uses the same 40,000 all-2025 flights. It overlaps development; it is
not another independent validation set. In each development fold, training precedes
April/July/October calibration, followed by two scoring months. The tables show those
boundaries and the earlier development evidence selecting Logistic Regression / B.

In [3]:
train = project.training_data()
y = train['delayed_departure'].astype(int)
fitted, seconds = {}, {}
display(project.cohort_table())

,fold,role,rows,first_date,last_date
0,May–Jun,train40,40000,2025-01-01,2025-03-31
1,May–Jun,calibrate,1574,2025-04-01,2025-04-30
2,May–Jun,score,3426,2025-05-01,2025-06-30
3,Aug–Sep,train40,40000,2025-01-01,2025-06-30
4,Aug–Sep,calibrate,1729,2025-07-01,2025-07-31
5,Aug–Sep,score,3271,2025-08-01,2025-09-30
6,Nov–Dec,train40,40000,2025-01-01,2025-09-30
7,Nov–Dec,calibrate,1781,2025-10-01,2025-10-31
8,Nov–Dec,score,3219,2025-11-01,2025-12-31


In [4]:
display(project.development_table())
display(project.threshold_table())

,model,B_package,mean_development_F1,mean_development_AP,settings
0,Logistic Regression,combined,0.483484,0.449869,{'C': 0.01}
1,KNN,combined,0.474153,0.416809,"{'n_neighbors': 101, 'weights': 'distance', 'm..."
2,SVM,current_66,0.447176,0.399481,"{'C': 3.0, 'gamma': 'scale'}"
3,Random Forest,pressure_wind,0.481618,0.436196,"{'max_depth': 16, 'min_samples_leaf': 5}"


,model,inputs,threshold
0,Logistic Regression,B,0.183914
1,Logistic Regression,A,0.146809
2,KNN,B,0.196502
3,KNN,A,0.159973
4,SVM,B,-0.890034
5,SVM,A,-2.408388
6,Random Forest,B,0.386966
7,Random Forest,A,0.465587


## Let scikit-learn handle preprocessing

`model_input` returns X in the frozen column order and an **unfitted sklearn Pipeline**.
Its `ColumnTransformer` uses `SimpleImputer`, `StandardScaler`, `OneHotEncoder` and
`MissingIndicator`. No fitting happens in the helper; every `.fit(X, y)` is below.

LR/KNN retain the separate continuous/binary/indicator recipe. SVM/RF retain the
original numeric/explicit-flag recipe; RF omits scaling. Each pipeline learns its
medians, scaling and categories from these 40,000 fitting rows only.
[Preprocessing implementation](../../scripts/final_modelling.py)

A has 26 inputs; B uses combined/78 for LR and KNN, current_66/66 for SVM, and
pressure_wind/72 for RF. The same classifier settings apply to A and B.

## Logistic Regression

A regularized linear boundary in the encoded features. C=.01 controls regularization.
The two fits learn separate preprocessing on identical rows. Elapsed time is recorded.

In [5]:
lr = LogisticRegression(C=.01, max_iter=3000, random_state=42)
for fs in ['A', 'B']:
    X, model = project.model_input(train, lr, 'logistic_regression', fs)
    started = perf_counter()
    fitted['logistic_regression__'+fs] = model.fit(X, y)
    seconds['logistic_regression__'+fs] = perf_counter()-started

## KNN

Predict from 101 nearby training flights using Euclidean distance and distance
weighting. Scaling matters because the original measurements use different units.
KNN stores training examples, so scoring can cost more than fitting.

In [6]:
knn = KNeighborsClassifier(n_neighbors=101, weights='distance', metric='euclidean',
                          algorithm='brute', n_jobs=2)
for fs in ['A', 'B']:
    X, model = project.model_input(train, knn, 'knn', fs)
    started = perf_counter()
    fitted['knn__'+fs] = model.fit(X, y)
    seconds['knn__'+fs] = perf_counter()-started

## SVM

RBF similarity permits a nonlinear boundary. The fixed C=3 and gamma='scale' came
from development. SVM outputs margins, not probabilities, and uses a frozen margin
threshold. Its fitting step takes several minutes on this dataset.

In [7]:
svm = SVC(C=3, gamma='scale', kernel='rbf', cache_size=512, random_state=42)
for fs in ['A', 'B']:
    X, model = project.model_input(train, svm, 'svm', fs)
    started = perf_counter()
    fitted['svm__'+fs] = model.fit(X, y)
    seconds['svm__'+fs] = perf_counter()-started

## Random Forest

Average 200 trees with depth 16, minimum leaf size 5 and balanced class weights.
Numerical scaling is omitted. All four visible classifier definitions match the
published settings; the helper stops if a definition differs.

In [8]:
rf = RandomForestClassifier(n_estimators=200, max_depth=16, min_samples_leaf=5,
    max_features='sqrt', class_weight='balanced', random_state=42, n_jobs=2)
for fs in ['A', 'B']:
    X, model = project.model_input(train, rf, 'random_forest', fs)
    started = perf_counter()
    fitted['random_forest__'+fs] = model.fit(X, y)
    seconds['random_forest__'+fs] = perf_counter()-started

## Save the eight fits with their unchanged thresholds

October scores from 40k Jan–Sep fits chose maximum-F1 thresholds (largest threshold
breaks ties). These transfer to all-2025 refits without using July to adjust them.
The independent verifier checks their derivation. Saving below writes the already
fitted models separately from the published bundle and displays actual fitting times.

In [9]:
display(project.save_models(fitted, seconds))

,fit_seconds
model_inputs,
logistic_regression__A,0.149725
logistic_regression__B,0.384300
knn__A,0.084571
knn__B,0.191288
svm__A,60.745491
svm__B,248.706712
random_forest__A,2.992763
random_forest__B,7.714782


Continue to [04 — Evaluation and Findings](04_evaluation_and_findings.ipynb).